# Fish and Fly training
This is the main final training for YOLO model for my project Fish And Fly (v1 version).

## Steps involved:

1. I have total 15 classes, using Roboflow for data annotaions, creating version and exporting zipped and extracted data(4000images) to Google drive.

2. I will use Roboflow API key to download all annotated images from Roboflow -> Drive.

3. Then I will mount Drive in this Colab file and start training.

4. I am using checkpoint resumption technique in YOLO model training, so that I can reuse/ retrain the learned weight later, even when session time out and even if not uploaded in local.

5. I am using Colab T4 GPU for YOLO training. But I will verify model training in local too, with the same parameters.

### Step1: Mounting Google drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Step2: Exporting dataset from Roboflow -> Google Drive

In [ ]:
import os

project_path = "/content/drive/MyDrive/Fish_YOLO_V1"
os.makedirs(project_path, exist_ok=True)

%cd /content/drive/MyDrive/Fish_YOLO_V1

/content/drive/MyDrive/Fish_YOLO_V1


In [ ]:
!pip install ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 149.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="aasxOHlFPuGPXLkIvX5g")
project = rf.workspace("1conviniencestore").project("marine-project-uoi6r")
version = project.version(3)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Marine-project-3 in yolov8:: 100%|██████████| 7317/7317 [01:12<00:00, 101.55it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### Checking if dataset is uploaded successfully in drive

In [ ]:
!ls Marine-project-3

data.yaml  README.dataset.txt  README.roboflow.txt  train  valid


In [ ]:
!ls Marine-project-3/train/images | wc -l

2927


In [2]:
%cd /content/drive/MyDrive/Fish_YOLO_V1

/content/drive/MyDrive/Fish_YOLO_V1


In [7]:
cat Marine-project-3/data.yaml

names:
- algae_bloom
- big_rock
- crab
- dangerous_animal
- fish
- fishing_net
- fishing_rope
- glass_bottle
- human
- jellyfish
- metal_can
- plastic_bag
- plastic_bottle
- starfish
- turtle
nc: 15
roboflow:
  license: CC BY 4.0
  project: marine-project-uoi6r
  url: https://universe.roboflow.com/1conviniencestore/marine-project-uoi6r/dataset/3
  version: 3
  workspace: 1conviniencestore
test: ../test/images
train: ../train/images
val: ../valid/images


In [8]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.5 MB/s eta 0:00:00


### Step3: YOLO Model training

In [9]:
import os
from ultralytics import YOLO

checkpoint_path = "/content/drive/MyDrive/Fish_YOLO_V1/Fish_V1/weights/last.pt"

if os.path.exists(checkpoint_path):
    print("Resuming from checkpoint...")
    model = YOLO(checkpoint_path)
    model.train(resume=True)
else:
    print("Starting fresh training...")
    model = YOLO("yolov8s.pt")
    model.train(
        data="/content/drive/MyDrive/Fish_YOLO_V1/Marine-project-3/data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        patience=30,
        optimizer="AdamW",
        lr0=0.001,
        project="/content/drive/MyDrive/Fish_YOLO_V1",
        name="Fish_V1",
        device=0
    )

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Starting fresh training...
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Fish_YOLO_V1/Marine-project-3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, ha

### NOTE: I pasted below the exact info, I got during model training, i. e. info of all the 100 epochs

```
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Starting fresh training...
Downloading https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8s.pt to 'yolov8s.pt': 100% ━━━━━━━━━━━━ 21.5MB 148.8MB/s 0.1s
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Fish_YOLO_V1/Marine-project-3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Fish_V1, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=30, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=/content/drive/MyDrive/Fish_YOLO_V1, rect=False, resume=False, retina_masks=False, rle=1.0, save=True, save_conf=False, save_crop=False, save_dir=/content/drive/MyDrive/Fish_YOLO_V1/Fish_V1, save_frames=False, save_json=False, save_period=-1, save_txt=False, scale=0.5, seed=0, shear=0.0, show=False, show_boxes=True, show_conf=True, show_labels=True, simplify=True, single_cls=False, source=None, split=val, stream_buffer=False, task=detect, time=None, tracker=botsort.yaml, translate=0.1, val=True, verbose=True, vid_stride=1, visualize=False, warmup_bias_lr=0.1, warmup_epochs=3.0, warmup_momentum=0.8, weight_decay=0.0005, workers=8, workspace=None
Downloading https://ultralytics.com/assets/Arial.ttf to '/root/.config/Ultralytics/Arial.ttf': 100% ━━━━━━━━━━━━ 755.1KB 118.6MB/s 0.0s
Overriding model.yaml nc=80 with nc=15

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytics.nn.modules.conv.Conv             [256, 512, 3, 2]              
  8                  -1  1   1838080  ultralytics.nn.modules.block.C2f             [512, 512, 1, True]           
  9                  -1  1    656896  ultralytics.nn.modules.block.SPPF            [512, 512, 5]                 
 10                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 11             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 12                  -1  1    591360  ultralytics.nn.modules.block.C2f             [768, 256, 1]                 
 13                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 14             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 15                  -1  1    148224  ultralytics.nn.modules.block.C2f             [384, 128, 1]                 
 16                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
 17            [-1, 12]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 18                  -1  1    493056  ultralytics.nn.modules.block.C2f             [384, 256, 1]                 
 19                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
 20             [-1, 9]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 21                  -1  1   1969152  ultralytics.nn.modules.block.C2f             [768, 512, 1]                 
 22        [15, 18, 21]  1   2121853  ultralytics.nn.modules.head.Detect           [15, 16, None, [128, 256, 512]]
Model summary: 130 layers, 11,141,405 parameters, 11,141,389 gradients, 28.7 GFLOPs
```

```
Transferred 349/355 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
Downloading https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt to 'yolo26n.pt': 100% ━━━━━━━━━━━━ 5.3MB 80.3MB/s 0.1s
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.5±0.2 ms, read: 0.1±0.0 MB/s, size: 43.1 KB)
train: Scanning /content/drive/MyDrive/Fish_YOLO_V1/Marine-project-3/train/labels... 2927 images, 606 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2927/2927 2.0it/s 24:52
train: New cache created: /content/drive/MyDrive/Fish_YOLO_V1/Marine-project-3/train/labels.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.9±0.5 ms, read: 0.1±0.0 MB/s, size: 66.7 KB)
val: Scanning /content/drive/MyDrive/Fish_YOLO_V1/Marine-project-3/valid/labels... 727 images, 147 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 727/727 2.0it/s 6:01
val: New cache created: /content/drive/MyDrive/Fish_YOLO_V1/Marine-project-3/valid/labels.cache
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Plotting labels to /content/drive/MyDrive/Fish_YOLO_V1/Fish_V1/labels.jpg... 
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/drive/MyDrive/Fish_YOLO_V1/Fish_V1
```


```
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      3.54G      1.422      3.455      1.661         83        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:09
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.2it/s 10.4s
                   all        727       1144      0.445      0.133     0.0854      0.042

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      4.32G      1.541      2.978      1.757         37        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.6s
                   all        727       1144      0.544      0.105     0.0964     0.0523

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      4.32G      1.535      2.986      1.767         55        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:08
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.6s
                   all        727       1144       0.35      0.179      0.119     0.0533

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      4.32G       1.53      2.896      1.758         27        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.3it/s 9.8s
                   all        727       1144      0.404      0.188      0.137     0.0666

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      5/100      4.32G      1.519      2.836      1.728         41        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.7s
                   all        727       1144      0.414      0.198      0.164     0.0861

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      6/100      4.32G      1.479      2.686      1.709         29        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.5it/s 9.3s
                   all        727       1144      0.581      0.176      0.154     0.0871

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      7/100      4.32G      1.448      2.658      1.691         58        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.7s
                   all        727       1144      0.547      0.178      0.179      0.102

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      8/100      4.32G      1.408      2.566      1.667         64        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.4s
                   all        727       1144      0.572      0.171      0.207      0.124

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      9/100      4.32G      1.386      2.487      1.648         39        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.5s
                   all        727       1144       0.72      0.206      0.242       0.14

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     10/100      4.32G      1.374      2.488      1.648         31        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.6s
                   all        727       1144      0.554      0.228      0.266       0.16

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     11/100      4.32G       1.36      2.374      1.621         52        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.5s
                   all        727       1144      0.327      0.277      0.249      0.152

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     12/100      4.32G      1.352      2.312      1.621         38        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.6s
                   all        727       1144      0.473       0.23      0.265      0.159

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     13/100      4.32G      1.312      2.271      1.584         52        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.6s
                   all        727       1144      0.337      0.288      0.287      0.174

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     14/100      4.32G      1.289      2.191      1.563         51        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.6s
                   all        727       1144      0.357      0.331      0.309      0.185

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     15/100      4.32G      1.288      2.139      1.563         27        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.6s
                   all        727       1144      0.383       0.35       0.35      0.208

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     16/100      4.32G      1.267      2.123      1.552         94        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.4s
                   all        727       1144      0.481      0.321      0.328      0.209

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     17/100      4.32G      1.247      2.073      1.535         41        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.4s
                   all        727       1144      0.586      0.338      0.355      0.221

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     18/100      4.32G       1.26      2.028      1.529         47        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:08
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 7.9s
                   all        727       1144      0.468      0.307      0.334       0.21

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     19/100      4.32G      1.261      2.026      1.535         70        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.1s
                   all        727       1144      0.368      0.393      0.355      0.216

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     20/100      4.32G      1.214       1.91      1.491         40        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.4s
                   all        727       1144       0.53      0.336      0.381      0.238

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     21/100      4.32G      1.228      1.945      1.515         25        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.1s
                   all        727       1144      0.461      0.391       0.36       0.22

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     22/100      4.32G      1.215       1.94      1.498         46        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 7.9s
                   all        727       1144      0.541      0.351      0.372      0.237

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     23/100      4.32G      1.208      1.882        1.5         39        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.1s
                   all        727       1144       0.45      0.408      0.388      0.235

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     24/100      4.32G      1.186      1.833      1.487         44        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 8.1s
                   all        727       1144      0.457      0.366      0.372      0.236

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     25/100      4.32G      1.191      1.817      1.471         46        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.4s
                   all        727       1144      0.459      0.411      0.405       0.26

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     26/100      4.32G      1.151      1.737      1.464         84        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 7.8s
                   all        727       1144      0.397      0.468      0.413      0.259

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     27/100      4.32G      1.199      1.726      1.482         64        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 7.8s
                   all        727       1144      0.583      0.358      0.393      0.254

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     28/100      4.32G       1.18       1.72      1.469         31        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.2s
                   all        727       1144      0.536      0.407      0.439      0.275

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     29/100      4.32G      1.155      1.699      1.454         31        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:08
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.6it/s 8.7s
                   all        727       1144      0.557      0.391      0.424      0.269

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     30/100      4.32G      1.131      1.696      1.435         41        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.6it/s 8.9s
                   all        727       1144      0.525      0.428      0.437      0.279

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     31/100      4.32G      1.139      1.662      1.446         52        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.5it/s 9.2s
                   all        727       1144      0.533      0.426      0.442      0.291

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     32/100      4.32G      1.138      1.661      1.434         53        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.5s
                   all        727       1144      0.413      0.497      0.434      0.266

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     33/100      4.32G      1.139      1.623      1.444         34        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.6it/s 8.9s
                   all        727       1144      0.441      0.464      0.433      0.279

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     34/100      4.32G      1.114      1.588      1.425         36        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.6it/s 8.7s
                   all        727       1144      0.498      0.463      0.456       0.29

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     35/100      4.32G      1.119      1.569      1.425         35        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.6it/s 8.7s
                   all        727       1144      0.457      0.507      0.471      0.288

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     36/100      4.32G      1.107      1.535      1.417         61        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.5it/s 9.1s
                   all        727       1144       0.55      0.417      0.452      0.293

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     37/100      4.32G      1.095      1.506        1.4        102        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.6s
                   all        727       1144      0.564      0.464       0.48      0.312

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     38/100      4.32G      1.116      1.529      1.424         40        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.7s
                   all        727       1144      0.534      0.484      0.469      0.312

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     39/100      4.32G      1.073      1.482       1.39         69        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.3it/s 10.0s
                   all        727       1144      0.482      0.456      0.459      0.306

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     40/100      4.32G       1.09      1.466      1.396         43        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.7s
                   all        727       1144      0.588      0.454      0.481      0.312

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     41/100      4.32G      1.078      1.465      1.398         54        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.6it/s 8.8s
                   all        727       1144      0.594      0.484      0.506       0.33

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     42/100      4.32G      1.049      1.371      1.363         44        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:09
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0it/s 7.7s
                   all        727       1144      0.695      0.438       0.49      0.323

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     43/100      4.32G      1.061      1.425      1.388         32        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.4s
                   all        727       1144      0.567      0.464      0.486      0.313

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     44/100      4.32G      1.047       1.37      1.358         52        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.6it/s 9.0s
                   all        727       1144      0.644       0.43      0.498      0.325

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     45/100      4.32G      1.054      1.357      1.372         45        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.7s
                   all        727       1144      0.627      0.457      0.503      0.334

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     46/100      4.32G      1.047      1.366      1.363         45        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:09
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0it/s 7.7s
                   all        727       1144      0.653      0.439      0.515      0.335

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     47/100      4.32G      1.036      1.341      1.363         48        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:08
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0it/s 7.8s
                   all        727       1144      0.655      0.451      0.503      0.331

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     48/100      4.32G       1.02      1.312      1.347         39        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0it/s 7.7s
                   all        727       1144      0.661      0.474      0.517      0.341

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     49/100      4.32G      1.011      1.262      1.329         64        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0it/s 7.8s
                   all        727       1144      0.617      0.451      0.518      0.349

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     50/100      4.32G      1.024      1.281      1.339         54        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.1s
                   all        727       1144      0.645      0.442      0.505       0.33

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     51/100      4.32G      1.003      1.255      1.338         53        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.3s
                   all        727       1144      0.633      0.481      0.531      0.339

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     52/100      4.32G      1.006      1.244      1.332         22        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.3s
                   all        727       1144       0.65      0.471      0.517      0.335

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     53/100      4.32G     0.9891      1.248      1.322         81        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0it/s 7.7s
                   all        727       1144      0.608      0.491      0.509       0.33

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     54/100      4.32G      1.004      1.223       1.33         37        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.3s
                   all        727       1144      0.628       0.49      0.546      0.363

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     55/100      4.32G      1.008      1.215      1.334         31        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.3s
                   all        727       1144      0.673      0.441       0.51       0.34

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     56/100      4.32G     0.9873      1.218      1.326         36        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.7s
                   all        727       1144      0.677      0.453      0.524      0.344

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     57/100      4.32G      0.978      1.181      1.316         37        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.6s
                   all        727       1144       0.67       0.46      0.522      0.343

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     58/100      4.32G     0.9834      1.173      1.317         27        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:03
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.5it/s 9.4s
                   all        727       1144      0.585      0.466      0.519      0.343

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     59/100      4.32G     0.9664      1.169      1.297         47        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.4s
                   all        727       1144      0.598      0.494      0.527      0.345

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     60/100      4.32G     0.9587      1.143      1.291         55        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:03
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.4s
                   all        727       1144      0.616      0.509      0.538      0.356

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     61/100      4.32G     0.9495      1.109      1.286         50        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.5it/s 9.1s
                   all        727       1144      0.561      0.499      0.514      0.343

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     62/100      4.32G     0.9474      1.096      1.285         42        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:03
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0it/s 7.6s
                   all        727       1144      0.639      0.501      0.558      0.371

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     63/100      4.32G     0.9302      1.083      1.272         55        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 7.8s
                   all        727       1144      0.661      0.503      0.546      0.366

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     64/100      4.32G     0.9347      1.057       1.27         77        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.1s
                   all        727       1144       0.58      0.531      0.552       0.37

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     65/100      4.32G     0.9257      1.067      1.277         36        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:09
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.1s
                   all        727       1144      0.598      0.516      0.534      0.357

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     66/100      4.32G     0.9336      1.063      1.287         32        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 8.0s
                   all        727       1144      0.613      0.501      0.536      0.362

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     67/100      4.32G     0.9275      1.035      1.269         22        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 7.9s
                   all        727       1144      0.615      0.503      0.535      0.359

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     68/100      4.32G     0.9274      1.017      1.265         43        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.3s
                   all        727       1144      0.547      0.558      0.555      0.375

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     69/100      4.32G     0.9066      1.018      1.257         40        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:08
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.4s
                   all        727       1144      0.665      0.481      0.547      0.367

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     70/100      4.32G     0.9177      1.025      1.264         29        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 7.9s
                   all        727       1144      0.552      0.564      0.535      0.358

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     71/100      4.32G     0.8968     0.9821      1.249         57        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.3s
                   all        727       1144      0.641      0.553      0.556      0.372

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     72/100      4.32G     0.9066     0.9701      1.247         37        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:09
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 8.0s
                   all        727       1144      0.601      0.542      0.551      0.373

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     73/100      4.32G     0.8875     0.9601      1.244         65        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0it/s 7.7s
                   all        727       1144      0.612       0.52      0.548      0.365

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     74/100      4.32G      0.892     0.9517      1.249         36        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.2s
                   all        727       1144      0.577      0.549      0.555      0.373

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     75/100      4.32G     0.8789      0.923       1.24         62        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:03
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.5s
                   all        727       1144      0.587      0.541      0.546      0.376

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     76/100      4.32G     0.8636     0.9211      1.221         39        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.4s
                   all        727       1144      0.592       0.56      0.542      0.367

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     77/100      4.32G     0.8811     0.9269       1.23         40        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:03
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.5s
                   all        727       1144      0.648      0.502      0.562      0.379

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     78/100      4.32G     0.8753     0.9141      1.224         41        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.5it/s 9.3s
                   all        727       1144      0.686       0.51      0.555      0.371

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     79/100      4.32G     0.8522      0.887      1.222         47        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:03
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.7s
                   all        727       1144      0.592       0.56       0.57      0.385

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     80/100      4.32G     0.8471     0.8823      1.213         53        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.5it/s 9.4s
                   all        727       1144      0.579      0.517      0.552      0.377

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     81/100      4.32G     0.8474     0.8779      1.217         40        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:05
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.5it/s 9.3s
                   all        727       1144      0.622      0.535      0.559      0.373

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     82/100      4.32G     0.8419     0.8773      1.209         41        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.5s
                   all        727       1144      0.584      0.555       0.56      0.379

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     83/100      4.32G     0.8265     0.8447      1.202         37        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0it/s 7.8s
                   all        727       1144      0.631      0.529      0.562       0.38

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     84/100      4.32G     0.8403      0.864      1.209         50        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0it/s 7.7s
                   all        727       1144      0.604      0.568      0.584      0.388

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     85/100      4.32G     0.8344     0.8464      1.202         37        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0it/s 7.7s
                   all        727       1144      0.673       0.51      0.564      0.382

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     86/100      4.32G     0.8184     0.8243      1.189         31        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.1s
                   all        727       1144      0.603      0.563      0.573      0.385

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     87/100      4.32G     0.8315     0.8346      1.201         39        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.6s
                   all        727       1144      0.605      0.577      0.572      0.386

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     88/100      4.32G     0.8232     0.8155      1.195         50        640: 100% ━━━━━━━━━━━━ 183/183 2.7it/s 1:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.4s
                   all        727       1144      0.692      0.491      0.572      0.388

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     89/100      4.32G     0.7936     0.7971      1.178         51        640: 100% ━━━━━━━━━━━━ 183/183 2.8it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.5s
                   all        727       1144      0.587      0.556      0.557      0.378

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     90/100      4.32G      0.806     0.7945      1.185         40        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:03
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.5s
                   all        727       1144       0.63      0.534       0.57      0.393
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     91/100      4.32G     0.6931      0.635      1.134         18        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.5it/s 9.2s
                   all        727       1144      0.567      0.549      0.543      0.377

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     92/100      4.32G     0.6863     0.6029      1.122         21        640: 100% ━━━━━━━━━━━━ 183/183 3.0it/s 1:00
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 8.0s
                   all        727       1144      0.633      0.484      0.547      0.379

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     93/100      4.32G      0.669     0.5708      1.104         16        640: 100% ━━━━━━━━━━━━ 183/183 3.0it/s 1:01
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.6s
                   all        727       1144      0.578      0.548      0.558      0.382

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     94/100      4.32G     0.6599     0.5735      1.093         26        640: 100% ━━━━━━━━━━━━ 183/183 3.0it/s 1:00
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.7it/s 8.7s
                   all        727       1144      0.651      0.512      0.565       0.39

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     95/100      4.32G     0.6563     0.5546      1.101         29        640: 100% ━━━━━━━━━━━━ 183/183 3.0it/s 1:01
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.0it/s 7.6s
                   all        727       1144      0.625      0.501      0.554      0.387

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     96/100      4.32G     0.6438     0.5353      1.092         24        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:02
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.5it/s 9.3s
                   all        727       1144      0.594      0.529      0.555       0.38

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     97/100      4.32G      0.637     0.5458      1.092         42        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:03
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.5it/s 9.2s
                   all        727       1144      0.626      0.526      0.561      0.383

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     98/100      4.32G     0.6289      0.514      1.074         23        640: 100% ━━━━━━━━━━━━ 183/183 2.9it/s 1:03
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.9it/s 8.1s
                   all        727       1144      0.598      0.542      0.559      0.385

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     99/100      4.32G     0.6319      0.521      1.079         16        640: 100% ━━━━━━━━━━━━ 183/183 3.0it/s 1:02
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.3s
                   all        727       1144       0.63      0.512      0.561      0.386

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    100/100      4.32G     0.6315     0.5216      1.085         19        640: 100% ━━━━━━━━━━━━ 183/183 3.0it/s 1:02
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.4it/s 9.6s
                   all        727       1144       0.56      0.582      0.565      0.387

100 epochs completed in 2.078 hours.
Optimizer stripped from /content/drive/MyDrive/Fish_YOLO_V1/Fish_V1/weights/last.pt, 22.5MB
Optimizer stripped from /content/drive/MyDrive/Fish_YOLO_V1/Fish_V1/weights/best.pt, 22.5MB
```

```
Validating /content/drive/MyDrive/Fish_YOLO_V1/Fish_V1/weights/best.pt...
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,131,389 parameters, 0 gradients, 28.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.3it/s 10.2s
                   all        727       1144       0.63      0.534       0.57      0.393
           algae_bloom          5          6       0.28      0.333      0.271      0.153
              big_rock         23         29      0.435      0.371      0.275      0.158
                  crab         44         53      0.712      0.421      0.586      0.368
      dangerous_animal         53         93      0.527      0.581      0.537       0.35
                  fish        124        355      0.637      0.499      0.561      0.377
           fishing_net         40         42      0.583      0.452      0.513      0.276
          fishing_rope          7         13      0.391      0.154      0.216      0.126
          glass_bottle         24         37      0.656      0.514      0.555       0.38
                 human         50         79      0.653      0.633      0.626      0.433
             jellyfish         29         69      0.718      0.522      0.617      0.393
             metal_can          8         29      0.612      0.652      0.679      0.482
           plastic_bag        100        142      0.816      0.655      0.724       0.51
        plastic_bottle         49         75      0.572      0.507      0.536      0.378
              starfish         45         49      0.977      0.875       0.95      0.808
                turtle         73         73      0.886      0.836      0.905      0.702
Speed: 0.2ms preprocess, 4.4ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to /content/drive/MyDrive/Fish_YOLO_V1/Fish_V1
```